# Search and screening audit

Checks required search-log fields and calculates PRISMA-informed screening counts from recorded decisions.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
searches = pd.read_csv(ROOT / "literature" / "search_log.csv", dtype=str).fillna("")
screening = pd.read_csv(ROOT / "literature" / "screening.csv", dtype=str).fillna("")

required = ["search_id", "search_type", "executed_date", "database", "exact_query", "result_count", "searcher"]
missing = {col: int(searches[col].eq("").sum()) for col in required}
invalid_counts = pd.to_numeric(searches["result_count"], errors="coerce").isna() & searches["result_count"].ne("")
print("Search rows:", len(searches))
print("Missing required values:", missing)
print("Non-numeric result_count rows:", int(invalid_counts.sum()))

counts = {
    "records_in_screening_file": len(screening),
    "duplicate_records_flagged": int(screening["duplicate_of"].ne("").sum()),
    "title_abstract_screened": int(screening["title_abstract_decision"].isin(["include", "exclude", "maybe"]).sum()),
    "title_abstract_excluded": int(screening["title_abstract_decision"].eq("exclude").sum()),
    "full_text_not_retrieved": int(screening["full_text_status"].eq("not_retrieved").sum()),
    "full_text_assessed": int(screening["full_text_decision"].isin(["include", "exclude", "maybe"]).sum()),
    "full_text_excluded": int(screening["full_text_decision"].eq("exclude").sum()),
    "included_studies": int(screening["full_text_decision"].eq("include").sum()),
}
pd.Series(counts, name="count").to_frame()